# WikiNews NLP Project — Text Summarization

This notebook applies extractive text summarization to the 60-article WikiNews analysis sample.

The main goals are to:

- reduce long news articles while preserving key facts,
- summarize 15 articles from each of the four selected categories,
- measure summary length and compression,
- inspect representative summaries,
- evaluate grammar and style characteristics,
- prepare summaries for semantic similarity analysis.

## 1. Imports and Data Loading

In [18]:
from pathlib import Path
import re

import networkx as nx
import pandas as pd
import spacy
from sklearn.feature_extraction.text import TfidfVectorizer

PROJECT_ROOT = Path("..").resolve()

nlp = spacy.load("en_core_web_sm")

In [19]:
DATA_PATH = PROJECT_ROOT / "data/processed/analysis_sample.csv"

analysis_df = pd.read_csv(DATA_PATH)

print("Dataset shape:", analysis_df.shape)
print("\nCategory distribution:")
print(analysis_df["primary_category"].value_counts())

Dataset shape: (60, 9)

Category distribution:
primary_category
Politics and conflicts    15
Economy and business      15
Science and technology    15
Sports                    15
Name: count, dtype: int64


## 2. Article Lengths

Before summarization, the word-count distribution of the selected articles is inspected.

In [20]:
analysis_df["word_count"].describe()

count      60.000000
mean      347.700000
std       332.949902
min       104.000000
25%       192.250000
50%       265.000000
75%       356.500000
max      2380.000000
Name: word_count, dtype: float64

In [21]:
analysis_df.groupby(
    "primary_category"
)["word_count"].agg(
    ["count", "mean", "median", "min", "max"]
).round(1)

,count,mean,median,min,max
primary_category,,,,,
Economy and business,15,310.1,224.0,126,701
Politics and conflicts,15,463.0,294.0,113,2380
Science and technology,15,299.1,256.0,184,794
Sports,15,318.7,263.0,104,1171


## 3. Extractive Summarization Method

An extractive summarizer is used to select the most informative sentences from each article.

The summarizer:

1. splits an article into sentences,
2. represents sentences using TF-IDF,
3. builds a sentence-similarity graph,
4. ranks sentences with PageRank,
5. selects the highest-ranked sentences,
6. restores the original sentence order for readability.

In [22]:
def split_sentences(text):
    """Split article text into sentences using spaCy."""
    doc = nlp(text)

    return [
        sentence.text.strip()
        for sentence in doc.sents
        if sentence.text.strip()
    ]

In [23]:
def summarize_text(
    text,
    compression_ratio=0.30,
    min_sentences=2,
):
    """Create an extractive summary using TF-IDF and PageRank."""
    sentences = split_sentences(text)

    if len(sentences) <= min_sentences:
        return text

    vectorizer = TfidfVectorizer(
        stop_words="english"
    )

    sentence_matrix = vectorizer.fit_transform(
        sentences
    )

    similarity_matrix = (
        sentence_matrix
        @ sentence_matrix.T
    ).toarray()

    graph = nx.from_numpy_array(
        similarity_matrix
    )

    scores = nx.pagerank(graph)

    ranked = sorted(
        scores.items(),
        key=lambda item: item[1],
        reverse=True,
    )

    summary_size = max(
        min_sentences,
        round(
            len(sentences)
            * compression_ratio
        ),
    )

    summary_size = min(
        summary_size,
        len(sentences),
    )

    selected_indices = sorted(
        index
        for index, _ in ranked[
            :summary_size
        ]
    )

    summary = " ".join(
        sentences[index]
        for index in selected_indices
    )

    return summary

In [24]:
sample_article = analysis_df.iloc[0]

sample_summary = summarize_text(
    sample_article["text"]
)

print("TITLE:")
print(sample_article["title"])

print("\nORIGINAL WORDS:")
print(len(sample_article["text"].split()))

print("\nSUMMARY WORDS:")
print(len(sample_summary.split()))

print("\nSUMMARY:")
print(sample_summary)

TITLE:
Brazilian President party received money from FARC, say documents

ORIGINAL WORDS:
2380

SUMMARY WORDS:
950

SUMMARY:
Brazil —
Documents of the Brazilian Agency of intelligence (Abin) say that the Workers' Party received 5 million dollars to be used by political campaign of  candidates in 2002 from the Colombian communist armed group Revolutionary Armed Forces of Colombia (FARC-EP). The information was reported by the Brazilian magazine Veja that circulates this week in a headline story called "FARC's tentacles in Brazil". According to the magazine, reporters had access to documents of Abin that described the liaisons  between the Workers' Party (PT) and the Colombian guerrilla movement FARC. The Workers' Party (PT) is the party of the Brazilian President Luiz Inácio Lula da Silva. PT is one of the biggest left-wing parties in Latin America and at this moment the strongest and more organized party from Brazil. Abin's main document, number 0095/3100 of April 25, 2002, says that a

## 4. Generate Summaries for All Selected Articles

The same extractive summarization process is applied to all 60 selected articles.

In [25]:
summaries_df = analysis_df.copy()

summaries_df["summary"] = (
    summaries_df["text"].apply(
        summarize_text
    )
)

summaries_df["original_word_count"] = (
    summaries_df["text"]
    .str.split()
    .str.len()
)

summaries_df["summary_word_count"] = (
    summaries_df["summary"]
    .str.split()
    .str.len()
)

summaries_df["compression_ratio"] = (
    summaries_df["summary_word_count"]
    / summaries_df["original_word_count"]
)

print("Summarized articles:", len(summaries_df))

Summarized articles: 60


In [26]:
summaries_df[
    [
        "original_word_count",
        "summary_word_count",
        "compression_ratio",
    ]
].describe().round(3)

,original_word_count,summary_word_count,compression_ratio
count,60.00,60.000,60.000
mean,347.70,115.850,0.330
std,332.95,124.969,0.064
min,104.00,30.000,0.179
25%,192.25,63.000,0.289
50%,265.00,82.500,0.328
75%,356.50,119.250,0.372
max,2380.00,950.000,0.526


In [27]:
summaries_df.groupby(
    "primary_category"
)[
    [
        "original_word_count",
        "summary_word_count",
        "compression_ratio",
    ]
].mean().round(3)

,original_word_count,summary_word_count,compression_ratio
primary_category,,,
Economy and business,310.067,108.267,0.354
Politics and conflicts,463.000,162.667,0.339
Science and technology,299.067,95.933,0.318
Sports,318.667,96.533,0.310


## 5. Example Summaries

Representative summaries are inspected to assess whether the main factual content is retained.

In [28]:
for category in summaries_df[
    "primary_category"
].unique():
    row = summaries_df.loc[
        summaries_df[
            "primary_category"
        ] == category
    ].iloc[0]

    print("=" * 80)
    print("CATEGORY:", category)
    print("TITLE:", row["title"])
    print("\nSUMMARY:")
    print(row["summary"])
    print()

CATEGORY: Politics and conflicts
TITLE: Brazilian President party received money from FARC, say documents

SUMMARY:
Brazil —
Documents of the Brazilian Agency of intelligence (Abin) say that the Workers' Party received 5 million dollars to be used by political campaign of  candidates in 2002 from the Colombian communist armed group Revolutionary Armed Forces of Colombia (FARC-EP). The information was reported by the Brazilian magazine Veja that circulates this week in a headline story called "FARC's tentacles in Brazil". According to the magazine, reporters had access to documents of Abin that described the liaisons  between the Workers' Party (PT) and the Colombian guerrilla movement FARC. The Workers' Party (PT) is the party of the Brazilian President Luiz Inácio Lula da Silva. PT is one of the biggest left-wing parties in Latin America and at this moment the strongest and more organized party from Brazil. Abin's main document, number 0095/3100 of April 25, 2002, says that a meeting 

## 6. Grammar and Style Checks

A lightweight automated review is used to identify potential grammar and style issues in the extractive summaries.

The checks focus on:

- incomplete sentence endings,
- very short summaries,
- repeated sentences,
- excessive sentence length,
- basic readability indicators.

These checks are heuristic and are not treated as a full grammar evaluation.

In [29]:
def style_checks(summary):
    """Return simple grammar and style indicators."""
    sentences = split_sentences(summary)

    issues = []

    if len(summary.split()) < 40:
        issues.append("Very short summary")

    if summary and summary[-1] not in ".!?":
        issues.append(
            "Summary may end with an incomplete sentence"
        )

    if len(sentences) != len(set(sentences)):
        issues.append(
            "Repeated sentence detected"
        )

    long_sentences = [
        sentence
        for sentence in sentences
        if len(sentence.split()) > 45
    ]

    if long_sentences:
        issues.append(
            "Contains very long sentence"
        )

    return "; ".join(issues)

In [30]:
summaries_df["style_issues"] = (
    summaries_df["summary"].apply(
        style_checks
    )
)

summaries_df["style_issue_flag"] = (
    summaries_df["style_issues"].ne("")
)

print(
    summaries_df[
        "style_issue_flag"
    ].value_counts()
)

summaries_df.loc[
    summaries_df["style_issue_flag"],
    [
        "title",
        "primary_category",
        "style_issues",
    ],
].head(10)

style_issue_flag
False    43
True     17
Name: count, dtype: int64


,title,primary_category,style_issues
3,Kurds announce deal with Assad's government as...,Politics and conflicts,Contains very long sentence
11,Nigeria hands over disputed area to Cameroon,Politics and conflicts,Contains very long sentence
12,Howard Dean elected chairman of U.S. Democrats,Politics and conflicts,Summary may end with an incomplete sentence
16,IMF head remains in New York prison; charged o...,Economy and business,Contains very long sentence
19,EU fines Microsoft €280.5 million,Economy and business,Very short summary
22,Romania redenominates its currency,Economy and business,Contains very long sentence
23,Market maker Bernard L. Madoff arrested in $50...,Economy and business,Contains very long sentence
35,Japan earthquake shifts Earth's axis 10 centim...,Science and technology,Contains very long sentence
37,Study: Taste of beer causes chemical reward in...,Science and technology,Summary may end with an incomplete sentence
39,Wikipedia founder Jimmy Wales to start wiki-ba...,Science and technology,Summary may end with an incomplete sentence


## 7. Save Summarization Results

The generated summaries and compression metrics are saved for semantic similarity analysis in the next notebook.

In [31]:
SUMMARY_PATH = (
    PROJECT_ROOT
    / "data/processed/summaries.csv"
)

columns_to_save = [
    "pageid",
    "title",
    "date",
    "primary_category",
    "url",
    "text",
    "summary",
    "original_word_count",
    "summary_word_count",
    "compression_ratio",
    "style_issues",
    "style_issue_flag",
]

summaries_df[
    columns_to_save
].to_csv(
    SUMMARY_PATH,
    index=False,
)

print("Saved:", SUMMARY_PATH)
print(
    "Shape:",
    summaries_df[
        columns_to_save
    ].shape,
)

Saved: /workspaces/rigoel-NLP.AI.2.5/data/processed/summaries.csv
Shape: (60, 12)


In [32]:
saved_summaries = pd.read_csv(
    SUMMARY_PATH
)

print(saved_summaries.shape)

saved_summaries[
    [
        "title",
        "primary_category",
        "original_word_count",
        "summary_word_count",
        "compression_ratio",
    ]
].head()

(60, 12)


,title,primary_category,original_word_count,summary_word_count,compression_ratio
0,Brazilian President party received money from ...,Politics and conflicts,2380,950,0.399160
1,Burundian Hutu extremists have killed 300 civi...,Politics and conflicts,326,96,0.294479
2,Spanish Socialist Workers' Party proposes huma...,Politics and conflicts,354,146,0.412429
3,Kurds announce deal with Assad's government as...,Politics and conflicts,361,120,0.332410
4,IRA disbands military structure,Politics and conflicts,639,225,0.352113


## Key Findings

- Extractive summarization was applied to all 60 selected WikiNews articles.
- The method uses sentence-level TF-IDF similarity and PageRank to identify informative sentences.
- The selected sentences are restored to their original order to preserve readability and narrative coherence.
- Summary length is substantially reduced compared with the original articles.
- Compression ratios provide a quantitative measure of how much each article was shortened.
- Lightweight grammar and style checks were used to flag potentially short, repetitive, incomplete, or overly long summaries.
- The generated summaries are saved for semantic similarity evaluation in the next stage.